Notebook réservé aux regressions (précédemment sur R, adapté sur Python avec statsmodel) 

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import os
import numpy as np

# chemin relatif qui respecte l'arborescence du Git (pour rendre réplicable)
file_path = os.path.join("..", "data", "merged", "data_regression_std.csv") # prendre les données standardisées!

# Chargement
df = pd.read_csv(file_path)



REGRESSION MCO 

In [36]:

# Régression Linéaire (OLS)

# Adapter la formule pour changer les variables Y ou explicatives X si besoin 

formula = "score_gauche_2020 ~ score_gauche_2014 + log_pop_2020 + log_med_19 + pct_sup_2020"

print(f"Régression sur la formule : {formula}")

# On lance le modèle (missing='drop' ignore juste les lignes avec des NaN résiduels s'il y en a)
model = smf.ols(formula=formula, data=df, missing='drop').fit(cov_type='HC1')
    
# summary du modèle 
print(model.summary())



Régression sur la formule : score_gauche_2020 ~ score_gauche_2014 + log_pop_2020 + log_med_19 + pct_sup_2020
                            OLS Regression Results                            
Dep. Variable:      score_gauche_2020   R-squared:                       0.174
Model:                            OLS   Adj. R-squared:                  0.172
Method:                 Least Squares   F-statistic:                     99.52
Date:                Sat, 20 Dec 2025   Prob (F-statistic):           2.38e-75
Time:                        12:25:14   Log-Likelihood:                -7098.4
No. Observations:                1533   AIC:                         1.421e+04
Df Residuals:                    1528   BIC:                         1.423e+04
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
---------------

code ci dessous vibe-codé (à revoir donc, je fais pas trop confiance)

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import os
import numpy as np

# --- 1. Chargement des données standardisées ---
# (Assure-toi que ce fichier contient bien les scores bruts non-standardisés comme prévu)
file_path = os.path.join("..", "data", "merged", "data_regression_std.csv")

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Données chargées : {len(df)} communes.")
else:
    raise FileNotFoundError(f"Fichier introuvable : {file_path}")

# --- 2. Le Test "Technique vs Politique" (Filtrage) ---
# On ne garde que les communes où il y a eu une liste de gauche les deux années.
# Cela élimine le bruit technique des "disparitions" ou "apparitions" de listes.
df_stable = df[(df['score_gauche_2014'] > 0) & (df['score_gauche_2020'] > 0)].copy()

print(f"--- FILTRAGE ---")
print(f"Communes totales : {len(df)}")
print(f"Communes avec offre stable (Test) : {len(df_stable)}")
print(f"Perte de {len(df) - len(df_stable)} communes instables.")

# --- 3. Régression sur l'Offre Stable ---
# Formule finale validée (avec les proportions pct_ et les logs)
formula = "score_gauche_2020 ~ score_gauche_2014 + log_pop_2020 + log_med_19 + pct_sup_2020"

print(f"\n--- RÉSULTATS DU TEST ---")
print(f"Formule : {formula}")

try:
    model_stable = smf.ols(formula=formula, data=df_stable, missing='drop').fit(cov_type='HC1')
    print(model_stable.summary())
    
    # Petit bonus d'interprétation automatique
    coeff_2014 = model_stable.params['score_gauche_2014']
    print("\n--- VERDICT ---")
    if coeff_2014 > 0:
        print(f" Coefficient 2014 positif ({coeff_2014:.3f}) : C'était bien un biais technique !")
        print("L'autocorrélation est rétablie : les bastions de 2014 restent des bastions.")
    else:
        print(f" Coefficient 2014 toujours négatif ({coeff_2014:.3f}) : C'est un fait POLITIQUE.")
        print("Les bastions de 2014 continuent de s'effondrer structurellement, même à offre constante.")

except Exception as e:
    print(f"Erreur lors de la régression : {e}")

Données chargées : 1533 communes.
--- FILTRAGE ---
Communes totales : 1533
Communes avec offre stable (Test) : 371
Perte de 1162 communes instables.

--- RÉSULTATS DU TEST ---
Formule : score_gauche_2020 ~ score_gauche_2014 + log_pop_2020 + log_med_19 + pct_sup_2020
                            OLS Regression Results                            
Dep. Variable:      score_gauche_2020   R-squared:                       0.360
Model:                            OLS   Adj. R-squared:                  0.353
Method:                 Least Squares   F-statistic:                     49.78
Date:                Sat, 20 Dec 2025   Prob (F-statistic):           1.95e-33
Time:                        12:28:55   Log-Likelihood:                -1572.7
No. Observations:                 371   AIC:                             3155.
Df Residuals:                     366   BIC:                             3175.
Df Model:                           4                                         
Covariance Type:      

Méthodoligie et résultats: synthèse de ce notebook et du notebook préparation des données

Analyse de régression visant à expliquer l'évolution du vote de gauche (2014-2020) par des déterminants sociologiques. Voici les points clés de la démarche :


1. Pré-traitement et standardisation

Transformation logarithmique : Les variables de richesse et de population ont été passées au logarithme avant toute standardisation pour éviter la création de valeurs manquantes (NaN) sur des valeurs centrées-réduites négatives

Choix des Variables : Utilisation stricte de taux (%) (Cadres, Diplômés) plutôt que de valeurs absolues pour éviter une multicolinéarité forte avec la variable de population.

Standardisation (Z-Score) : Appliquée uniquement aux variables explicatives (X) pour comparer les coefficients (coeffs beta), mais pas à la variable cible (Y = score gauche année X ou delta score) pour conserver l'interprétation en points de pourcentage de voix.



2. Le "paradoxe" du score 2014 (Biais Technique vs Politique)

Observation initiale : Sur l'échantillon global (~1500 communes), le coefficient du score 2014 était négatif (-0.21). Cela suggérait un effondrement structurel des bastions historiques de gauche.

Hypothèse du Biais : Ce résultat était faussé par l'instabilité de l'offre politique locale (apparition/disparition de listes créant des scores de 0 artificiels).

Correction : En filtrant uniquement sur les communes avec une offre stable (présence de liste de gauche en 2014 ET 2020), le coefficient repasse à +0.67.

Conclusion : L'autocorrélation est rétablie. L'inertie du vote existe toujours ; l'anomalie était TECHNIQUE et non POLITIQUE.



3. Interprétation du Modèle Final (Offre Stable, N=371, R^2=0.36)

Le modèle final est robuste et confirme les dynamiques sociologiques attendues :

Inertie forte, le vote passé reste le premier déterminant du vote futur.

Effet Métropole (log_pop), Plus la commune est peuplée, plus le vote gauche résiste/progresse.

Clivage Culturel (pct_sup), À richesse égale, une forte part de diplômés favorise le vote de gauche (effet "bobo" / vote culturel).

Clivage Économique (log_med) : La richesse reste un frein massif au vote de gauche (coefficient négatif fort).